# Chapter 3 — Key Learnings

This notebook contains my main takeaways, definitions, and conceptual notes
from **Chapter 3 - Coding Attention Mechanisms** of *'Build a Large Language Model (From Scratch)'* book by Sebastian Raschka.

### Chapter Objective

Understand and implement the attention mechanism used in GPT-like language models, progressing from basic self-attention to causal multi-head attention.

### Conceptual Progression

```text
Input embeddings
→ simplified self-attention
→ trainable query, key, and value projections
→ scaled dot-product attention
→ causal masking
→ attention dropout
→ multi-head attention
→ context-aware token representations

### Scaled Dot-Product Attention

Given an input tensor $X$, self-attention creates three learned projections:

$$
\begin{aligned}
Q &= X \cdot W_Q \\
K &= X \cdot W_K \\
V &= X \cdot W_V
\end{aligned}
$$

The attention computation is:

$$
\begin{aligned}
\mathrm{Attention}(Q, K, V)
&=
\mathrm{softmax}
\left(
\frac{QK^{T}}{\sqrt{d_k}}
\right) \cdot V
\end{aligned}
$$

The steps are:

1. Compute query–key similarity scores.
2. Scale the scores by $\sqrt{d_k}$ .
3. Normalize them with softmax.
4. Use the resulting attention weights to combine the value vectors.

### Queries, Keys, and Values

- **Query:** what the current token is looking for.
- **Key:** what each token can be matched against (What each past token represents).
- **Value:** the information each token contributes.

A query is compared with all keys to obtain attention scores.  
The normalized scores determine how much of each value is included in the resulting context vector.

### **SelfAttention** class using PyTorch's Linear layers

In [1]:
import torch
import torch.nn as nn

class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # trainable weight matrix of dimensions (d_in, d_out)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias) # trainable weight matrix of dimensions (d_in, d_out)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) # trainable weight matrix of dimensions (d_in, d_out)
    
    def forward(self, x): # x has dim (n, d_in) where n:num_tokens
        queries = self.W_query(x) # this is equivalent to: Q = X . Wq (i.e. Projection Q and has dim (n, d_out) )
        keys = self.W_key(x) # this is equivalent to: K = X . Wk (i.e. Projection K and has dim (n, d_out) )
        values = self.W_value(x) # this is equivalent to: V = X . Wv (i.e. Projection V and has dim (n, d_out) )

        d_k = keys.shape[-1] # i.e. d_out
        attn_scores = queries @ keys.T # aka. similarity scores = Q.K^T # attn_scores dim: (n,n) i.e. attn_scores is a square matrix!
        attn_weights = torch.softmax(attn_scores/d_k**0.5, dim=-1) # i.e. softmax(Q.K^T/sqrt(d_k))
        context_vec = attn_weights @ values # i.e. the final Attention Matrix: Attention(Q, K, V) = softmax(Q.K^T/sqrt(d_k)) . V
        return context_vec

### **CausalAttention** class and associated **MultiHeadAttentionWrapper** class
The **MultiHeadAttentionWrapper** class although simple to implement, is not very efficient because it  *processes the CausalAttention Heads sequentially*.  
A better, more efficient class is the **MultiHeadAttention** class described later.

**Notes:**  

GPT generates text from left to right. Therefore, a token must not access tokens that occur later in the sequence.

A causal mask sets all attention scores above the main diagonal to negative infinity before softmax:

- Visible positions: current and previous tokens
- Masked positions: future tokens

After softmax, the masked positions receive an attention weight of zero.

**Dropout** is then applied to the remaining attention weights during training to reduce overfitting.

In [2]:
import torch.nn as nn

class CausalAttention(nn.Module): # The main ideas is that we apply a causal mask (to attn_scores) AND a dropout mask (to attn_weights) with prob p = dropout rate
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # trainable weight matrix of dimensions (d_in, d_out)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias) # trainable weight matrix of dimensions (d_in, d_out)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) # trainable weight matrix of dimensions (d_in, d_out)
        self.d_out = d_out
        self.dropout = nn.Dropout(dropout) # for the dropout mask with prob p = dropout rate - Note that the surviving nodes have a scaling factor of 1/(1-p)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # for the causal mask - corresponds to self.mask , register_buffer for gpu/cpu devices

    def forward(self, x): # x has dim (b, n, d_in) where b: batch_size,  n:num_tokens
        b, num_tokens, d_in = x.shape # Note that num_tokens <= context_length (context_length is the maximum sequence length, whereas num_tokens is the actual seq length in the batch)
        queries = self.W_query(x) # this is equivalent to: Q = X . Wq (i.e. Projection Q and has dim (b, n, d_out) )
        keys = self.W_key(x) # this is equivalent to: K = X . Wk (i.e. Projection K and has dim (b, n, d_out) )
        values = self.W_value(x) # this is equivalent to: V = X . Wv (i.e. Projection V and has dim (b, n, d_out) )
        d_k = keys.shape[-1] # i.e. d_out

        attn_scores = queries @ keys.transpose(1,2) # queries dim: (b,n,d_out) and keys.transpose(1,2) dim: (b, d_out,n) --> attn_scores dim: (b, n,n) i.e. a batch_size number of square matrices
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) # applying the causal mask: we use -inf in the upper triangular area via self.mask
        attn_weights = torch.softmax(attn_scores/d_k**0.5, dim=-1) # dim: (b, n,n)
        attn_weights = self.dropout(attn_weights) # applying the dropout mask to attn_weights
        context_vec = attn_weights @ values # context_vec dim: (b, n, d_out)
        return context_vec

class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1) 
        # dim=-1:concatenate along the last dim of the tensor. Ex: With 2 heads --> dim of results in Head 1: (b, n, d_out) in Head 2: (b, n, d_out) After concatenation: (b, n, 2 × d_out)


# Worthy of noting:
# Conceptually, each projection maps: # (d_in) -> (d_out)
# However, PyTorch stores each nn.Linear weight internally as: # (d_out, d_in)

The full shape flow of CausalAttention is:
```text
x
(b, n, d_in)

↓ W_query, W_key, W_value (d_in, d_out)

queries, keys, values (i.e. Q, K, V projections)
(b, n, d_out)

↓ QKᵀ

attn_scores
(b, n, n)

↓ masking, softmax, dropout

attn_weights
(b, n, n)

↓ attention weights × values

context_vec
(b, n, d_out)
```
where:  
b = batch size  
n = number of tokens  
d_in = input embedding dimension  
d_out = output embedding dimension  

### **MultiHeadAttention** class
The **MultiHeadAttention** class although a bit more complicated to implement than the **MultiHeadAttentionWrapper**, is very efficient because it  *processes the CausalAttention Heads in parallel* 
Efficient MHA computes the query, key, and value projections for all heads together and processes the heads using batched matrix multiplications.
> The wrapper invokes separate CausalAttention modules, whereas the efficient implementation projects once, splits the projected tensors into heads, and computes all heads through batched tensor operations.


In [ ]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert(d_out % num_heads == 0), "d_out must be divisible by num_heads"
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # trainable weight matrix of dimensions (d_in, d_out)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias) # trainable weight matrix of dimensions (d_in, d_out)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) # trainable weight matrix of dimensions (d_in, d_out)
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape # Note that num_tokens <= context_length (context_length is the maximum sequence length, whereas num_tokens is the actual seq length in the batch)
        queries = self.W_query(x) # this is equivalent to: Q = X . Wq (i.e. Projection Q and has dim (b, n, d_out) )
        keys = self.W_key(x) # this is equivalent to: K = X . Wk (i.e. Projection K and has dim (b, n, d_out) )
        values = self.W_value(x) # this is equivalent to: V = X . Wv (i.e. Projection V and has dim (b, n, d_out) )
        

        # Here, the last dim 'd_out' is now split into self.num_heads and self.head_dim (d_out = num_heads * head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim) # queries dim: (b, n, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) # keys dim: (b, n, num_heads, head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim) # values dim: (b, n, num_heads, head_dim)

        # Here, we swap n, num_heads dim positions so that we can keep (b, num_heads) fixed, while we operate matrix multiplications using the last 2 dims (n, head_dim)
        queries = queries.transpose(1,2) # queries dim: (b, num_heads, n, head_dim)
        keys = keys.transpose(1,2) # keys dim: (b, num_heads, n, head_dim)
        values = values.transpose(1,2) # values dim: (b, num_heads, n, head_dim)
        d_k = keys.shape[-1] # i.e. head_dim

        attn_scores = queries @ keys.transpose(2,3) # attn_scores is a (batch_size, num_heads) 'number' of square matrices --> dim: (b,num_heads, n,n)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) # applying the causal mask
        attn_weights = torch.softmax(attn_scores/d_k**0.5, dim=-1) # attn_weights dim: (b,num_heads, n,n)
        attn_weights = self.dropout(attn_weights) # applying the dropout mask --> # attn_weights dim: (b,num_heads, n,n)
        context_vec = (attn_weights @ values).transpose(1,2) # (attn_weights @ values) dim: (b,num_heads, n, head_dim) --> context_vec dim: (b,n, num_head, head_dim)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out) # UNSPLITTING: num_heads and head_dim are now combined BACK into d_out. 'contiguous()' ensures a compatible memory layout after transpose
        context_vec = self.out_proj(context_vec) # applies a learned linear transformation to the concatenated head outputs, allowing features from different heads to be mixed
        return context_vec # dim:(b, n, d_out)


## Efficient Multi-Head Attention Shape Flow

Let:

```text
h = number of heads
head_dim = d_out / h
```

The shape transformation is:

```text
Input
[b, n, d_in]

↓ shared Q, K, V projections

Q, K, V
[b, n, d_out]

↓ split d_out across heads (d_out = num_heads*head_dim)

[b, n, h, head_dim]

↓ transpose for batched attention

[b, h, n, head_dim]

↓ Q @ Kᵀ

attention scores
[b, h, n, n]

↓ mask, scale, softmax, dropout

attention weights
[b, h, n, n]

↓ attention weights @ V

context vectors per head
[b, h, n, head_dim]

↓ transpose and merge heads

combined context vectors
[b, n, d_out]

↓ output projection

final context vectors
[b, n, d_out]
```

### Output Projection

After the head outputs are combined, `out_proj` applies a learned linear
transformation:

```python
self.out_proj = nn.Linear(d_out, d_out)
```
This allows information from the different attention heads to be mixed before the result is passed to the next part of the transformer.

### Important Tensor Shapes

| Object | Shape |
|---|---|
| Input embeddings | `[b, n, d_in]` |
| Queries, keys, values before splitting | `[b, n, d_out]` |
| Queries, keys, values after splitting | `[b, h, n, head_dim]` |
| Attention scores | `[b, h, n, n]` |
| Attention weights | `[b, h, n, n]` |
| Context vectors per head | `[b, h, n, head_dim]` |
| Combined context vectors | `[b, n, d_out]` |
| Output-projected context vectors | `[b, n, d_out]` |

```text
d_out = num_heads × head_dim

### Snippets to test the above classes:

In [4]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [5]:
# We test the SelfAttention class
# We assume batch_size b=1
torch.manual_seed(123)
d_in = inputs.shape[1]
d_out = 2

sa = SelfAttention(d_in, d_out)
context_vecs = sa(inputs)
print(context_vecs.shape)
print(context_vecs)

torch.Size([6, 2])
tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


In [6]:
# simulating data that has batch-size b=2
batch = torch.stack((inputs, inputs), dim=0) 
print(batch)

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])


In [7]:
# We test the CausalAttention class
torch.manual_seed(123)
b, context_length, d_in = batch.shape
d_out =2
print(f"b, num_tokens, d_in, d_out: {b, context_length, d_in, d_out}\n")

ca = CausalAttention(d_in, d_out, context_length, dropout=0.5)
context_vecs =ca(batch)
print(context_vecs.shape)
print(context_vecs)


b, num_tokens, d_in, d_out: (2, 6, 3, 2)

torch.Size([2, 6, 2])
tensor([[[-0.9038,  0.4432],
         [-0.4368,  0.2142],
         [-0.4849, -0.1341],
         [-0.5834,  0.0081],
         [-0.6219, -0.0526],
         [-0.1417, -0.0505]],

        [[ 0.0000,  0.0000],
         [-1.1749,  0.0116],
         [-0.7733,  0.0073],
         [-0.9140, -0.2769],
         [-0.7679, -0.0735],
         [-0.6749, -0.0984]]], grad_fn=<UnsafeViewBackward0>)


In [8]:
# We test the MultiHeadAttentionWrapper class
torch.manual_seed(123)
b, context_length, d_in = batch.shape
d_out = 2
print(f"b, num_tokens, d_in, d_out: {b, context_length, d_in, d_out}\n")

mhaw = MultiHeadAttentionWrapper(d_in, d_out, context_length, dropout=0.2, num_heads=2)
context_vecs = mhaw(batch)
print(context_vecs.shape)
print(context_vecs)

b, num_tokens, d_in, d_out: (2, 6, 3, 2)

torch.Size([2, 6, 4])
tensor([[[-0.5649,  0.2770,  0.5965,  0.1329],
         [-0.7343,  0.0072,  0.2624,  0.0585],
         [-0.7875, -0.0790,  0.6065,  0.4448],
         [-0.7093, -0.1053,  0.5497,  0.4186],
         [-0.4859, -0.1402,  0.4938,  0.3032],
         [-0.5706, -0.1801,  0.5592,  0.3711]],

        [[ 0.0000,  0.0000,  0.5965,  0.1329],
         [-0.2730,  0.1339,  0.7363,  0.4071],
         [-0.7875, -0.0790,  0.3035,  0.2220],
         [-0.7093, -0.1053,  0.6847,  0.4487],
         [-0.5073, -0.0719,  0.3803,  0.2779],
         [-0.5104, -0.0930,  0.5538,  0.3622]]], grad_fn=<CatBackward0>)


In [9]:
# We test the MultiHeadAttention class
torch.manual_seed(123)
b, context_length, d_in = batch.shape
d_out = 2
print(f"b, num_tokens, d_in, d_out: {b, context_length, d_in, d_out}\n")

mha = MultiHeadAttention(d_in, d_out, context_length, dropout=0.2, num_heads=2)
context_vecs = mha(batch)
print(context_vecs.shape)
print(context_vecs)

b, num_tokens, d_in, d_out: (2, 6, 3, 2)

torch.Size([2, 6, 2])
tensor([[[0.2876, 0.4001],
         [0.3158, 0.3155],
         [0.2951, 0.4524],
         [0.2881, 0.3142],
         [0.2614, 0.3804],
         [0.2542, 0.4246]],

        [[0.3504, 0.4366],
         [0.3471, 0.3337],
         [0.2577, 0.4314],
         [0.2549, 0.4478],
         [0.2625, 0.3770],
         [0.2471, 0.4090]]], grad_fn=<ViewBackward0>)


### Key Takeaways

- Attention produces context-aware token representations.
- Attention scores measure query–key similarity.
- Softmax converts attention scores into normalized attention weights.
- The context vector is a weighted combination of value vectors.
- Query, key, and value projections are trainable.
- Scaling by the square root of the key dimension stabilizes softmax.
- Causal masking prevents access to future tokens.
- Dropout regularizes attention during training.
- Multi-head attention lets the model learn several relationship patterns.
- The efficient implementation computes all heads through batched matrix
  operations.
- The output projection mixes information from the combined heads.

## Q/As

- **Q: Describe the role of the encoder and decoder in a language translation model.**  
  The encoder processes the entire input text and encodes its meaning into a hidden state.  
  The decoder then uses this hidden state to generate the translated text, one word at a time.

- **Q: What is the primary limitation of encoder-decoder RNNs in handling long sequences?**  
  Encoder-decoder RNNs rely solely on the current hidden state during decoding, which can lead to a loss of context, especially when dependencies span long distances in complex sentences. 
  
- **Q: What is the significance of the `register_buffer` method in the CausalAttention class?**  
  The `register_buffer` method ensures that the causal mask is automatically moved to the appropriate device (CPU or GPU) along with the model, avoiding device mismatch errors during training. 
- **Q: What is the purpose of the output projection layer in the MultiHeadAttention class?**  
  The output projection layer applies a **learned linear transformation** to the concatenated head outputs, allowing features from different heads to be mixed

- **What is the difference between an attention score and an attention weight?**  
  A score is an unnormalized query–key similarity;   
  a weight is its normalized softmax value.

- **Why divide attention scores by the square root of the key dimension?**  
  It prevents large dot products from making softmax overly sharp and gradients unstable.

- **Why is a causal mask required in GPT?**  
  It prevents each token from accessing future tokens during next-token prediction.

- **Why must `d_out` be divisible by `num_heads`?**  
  Each head must receive an equal-sized portion of the output dimension.

- **What does `transpose(1, 2)` accomplish in multi-head attention?**  
  It places the head dimension before the token dimension so attention can be computed independently for every head.

- **What does `contiguous().view(...)` accomplish?**  
  It restores a compatible memory layout and merges the head dimensions back into `d_out`.
